# Orbit Ephemeris Message

An Orbit Ephemeris Message (OEM) is a table of position and velocity vectors of one object at a sequence of times, defined by the CCSDS *Orbit Data Messages* standard ([CCSDS 502.0-B-3](../../guide/references/#ccsds502)). It is the usual way operators and agencies exchange trajectories: NASA publishes the ISS trajectory as an OEM, Space-Track accepts operator ephemerides in the format for conjunction screening, and GMAT, STK and Orekit all read and write it.

An OEM is not an [OMM](../Orbital%20Mean-Element%20Message/). An OMM holds one set of mean elements and needs SGP4 to produce a state. An OEM holds the states themselves, so it is only interpolated, and it is as accurate as whatever produced it.

`satkit.OEM` reads both encodings of the format (KVN text and XML). Three things to know before using it:

- **Units.** The file gives kilometers and km/s. satkit converts to meters and m/s, as everywhere else.
- **Frame.** States stay in the frame the file names (`REF_FRAME`). Nothing is rotated on reading.
- **Segments.** A message is a list of segments, each with its own metadata. A producer starts a new segment where the states must not be interpolated across, such as a maneuver.

## Example 1: Reading an OEM

To have a file to read, this example first makes one: it propagates a low Earth orbit for three hours and writes the states every 60 seconds in the KVN encoding. The format is plain text: a short header, a metadata block between `META_START` and `META_STOP`, then one line per epoch with the position and velocity.

`sk.OEM.from_file` reads the file. The encoding is detected from the content, so the file extension does not matter.

In [ ]:
import os
import tempfile

import numpy as np
import satkit as sk

# A 500 km, 51.6 degree orbit, propagated for three hours
epoch = sk.time(2024, 6, 1)
kep = sk.kepler(sk.consts.earth_radius + 500e3, 0.001, np.radians(51.6), 0.0, 0.0, 0.0)
pos0, vel0 = kep.to_pv()
truth = sk.propagate(
    np.concatenate([pos0, vel0]),
    epoch,
    end=epoch + sk.duration(hours=3),
    propsettings=sk.propsettings(gravity_degree=20, abs_error=1e-12, rel_error=1e-12),
)

# Sample it every 60 seconds and write an OEM (kilometers and km/s)
step = 60
times = [epoch + sk.duration(seconds=s) for s in range(0, 3 * 3600 + 1, step)]
states = truth.interp(times)

lines = [
    "CCSDS_OEM_VERS = 2.0",
    "CREATION_DATE = 2024-06-01T00:00:00",
    "ORIGINATOR = SATKIT TUTORIAL",
    "",
    "META_START",
    "OBJECT_NAME = DEMOSAT",
    "OBJECT_ID = 2024-999A",
    "CENTER_NAME = EARTH",
    "REF_FRAME = GCRF",
    "TIME_SYSTEM = UTC",
    f"START_TIME = {times[0].strftime('%Y-%m-%dT%H:%M:%S')}",
    f"STOP_TIME = {times[-1].strftime('%Y-%m-%dT%H:%M:%S')}",
    "INTERPOLATION = LAGRANGE",
    "INTERPOLATION_DEGREE = 7",
    "META_STOP",
    "",
]
for t, s in zip(times, states):
    values = " ".join(f"{v / 1e3:16.9f}" for v in s)
    lines.append(f"{t.strftime('%Y-%m-%dT%H:%M:%S')} {values}")

path = os.path.join(tempfile.mkdtemp(), "demosat.oem")
with open(path, "w") as f:
    f.write("\n".join(lines) + "\n")

print("\n".join(lines[:19]))
print("...")

In [ ]:
oem = sk.OEM.from_file(path)

print(f"version {oem.version}, written by {oem.originator} on {oem.creation_date}")
print(f"{len(oem.segments)} segment(s)\n")

seg = oem.segments[0]
print(f"object:        {seg.object_name} ({seg.object_id})")
print(f"center, frame: {seg.center_name}, {seg.ref_frame}  ->  {seg.frame}")
print(f"time system:   {seg.time_system}")
print(f"span:          {seg.start_time} to {seg.stop_time}")
print(f"interpolation: {seg.interpolation}, degree {seg.interpolation_degree}")
print(f"states:        {len(seg)}")

# The states are numpy arrays in meters and m/s
print(f"\npos {seg.pos.shape}, vel {seg.vel.shape}")
print(f"first epoch:    {seg.time[0]}")
print(f"first position: {seg.pos[0] / 1e3} km")
print(f"first velocity: {seg.vel[0] / 1e3} km/s")

## Example 2: Interpolation

`interp` gives the state at any time the message covers. With no other argument it uses the method and degree the file recommends (`INTERPOLATION` and `INTERPOLATION_DEGREE`), here a 7th-degree Lagrange polynomial through the 8 nearest points. `method` and `degree` override the file:

- `"lagrange"` fits position and velocity separately, through `degree + 1` points.
- `"hermite"` fits position and velocity together, so it needs only `(degree + 1) / 2` points for the same degree. A file that names no method gets Hermite.

The plot compares the interpolated positions with the propagation the file was sampled from. The file was written with nine decimals of a kilometer, so nothing can agree to better than about a micrometer.

In [ ]:
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
plt.style.use(["science", "no-latex", "../satkit.mplstyle"])
%config InlineBackend.figure_formats = ['svg']

# One state, with the file's interpolation
t = epoch + sk.duration(minutes=47.5)
state = oem.interp(t)
print(f"position at {t}: {state[:3] / 1e3} km")
print(f"velocity:                              {state[3:] / 1e3} km/s")

# Many times at once: an (N, 6) array
fine = [epoch + sk.duration(seconds=s) for s in range(0, 3 * 3600 + 1, 7)]
reference = truth.interp(fine)[:, :3]
minutes = np.array([(t - epoch).minutes for t in fine])

cases = {
    "File default (Lagrange, degree 7)": {},
    "Hermite, degree 7": {"method": "hermite", "degree": 7},
    "Hermite, degree 3": {"method": "hermite", "degree": 3},
    "Lagrange, degree 3": {"method": "lagrange", "degree": 3},
    "Linear": {"method": "lagrange", "degree": 1},
}

fig, ax = plt.subplots(figsize=(10, 4.5))
for label, kwargs in cases.items():
    err = np.linalg.norm(oem.interp(fine, **kwargs)[:, :3] - reference, axis=1)
    ax.semilogy(minutes, np.maximum(err, 1e-7), label=label, linewidth=0.8)
    print(f"{label:36s} max error {err.max():12.6f} m")
ax.set_xlabel("Minutes from epoch")
ax.set_ylabel("Position error (m)")
ax.set_title("Interpolating a 60-second ephemeris")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

Linear interpolation of a 60-second ephemeris is wrong by kilometers, since the orbit curves about 4 km away from the chord between two points. The 7th-degree Lagrange fit is good to a tenth of a millimeter, and to about 2 mm at the two ends of the table, where the points can only be taken from one side. Hermite of the same degree is ten times better again, because it also uses the velocities. For a sparser ephemeris the gap between the two widens.

A time outside the span of the message is an error, as is a time in a gap between two segments: `interp` never extrapolates and never interpolates across segments.

In [ ]:
try:
    oem.interp(epoch + sk.duration(hours=4))
except ValueError as e:
    print(e)

## Example 3: Ground track

The states are in the frame the file names. `seg.frame` is the matching `satkit.frame` when satkit has one, which makes the conversion to another frame one call. Here the GCRF positions are rotated to the Earth-fixed ITRF and converted to latitude and longitude.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

assert seg.center_name == "EARTH" and seg.frame == sk.frame.GCRF

track_times = [epoch + sk.duration(seconds=s) for s in range(0, 3 * 3600 + 1, 20)]
pos = oem.interp(track_times)[:, :3]

quats = sk.frametransform.rotation(seg.frame, sk.frame.ITRF, track_times)
coords = [sk.itrfcoord(q * p) for q, p in zip(quats, pos)]
lat = np.array([c.latitude_deg for c in coords])
lon = np.array([c.longitude_deg for c in coords])

# Break the track where it crosses the date line
breaks = np.where(np.abs(np.diff(lon)) > 180)[0] + 1

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
for lo, la in zip(np.split(lon, breaks), np.split(lat, breaks)):
    ax.plot(lo, la, linewidth=1.5, color="C0", transform=ccrs.PlateCarree())
ax.set_title(f"{seg.object_name} ground track from an OEM")
ax.set_global()
plt.tight_layout()
plt.show()

## Example 4: Segments, time systems and covariance

`sk.OEM.from_text` parses text already in memory. The message below has the other features of the format:

- **Two segments**, split at a maneuver. Each has its own metadata, and `oem.interp` picks the segment that holds the requested time.
- **A usable span** (`USEABLE_START_TIME` / `USEABLE_STOP_TIME`) narrower than the data, which `interp` respects.
- **GPS time.** Epochs are read in the segment's `TIME_SYSTEM` and become ordinary `satkit.time` objects, so they compare correctly with any other time.
- **A covariance block**: the lower triangle of a 6x6 position and velocity covariance, here in the RTN frame.

In [ ]:
text = """CCSDS_OEM_VERS = 2.0
CREATION_DATE = 2024-06-01T00:00:00
ORIGINATOR = SATKIT TUTORIAL

META_START
OBJECT_NAME = DEMOSAT
OBJECT_ID = 2024-999A
CENTER_NAME = EARTH
REF_FRAME = EME2000
TIME_SYSTEM = GPS
START_TIME = 2024-06-01T00:00:00
USEABLE_START_TIME = 2024-06-01T00:01:00
USEABLE_STOP_TIME = 2024-06-01T00:02:00
STOP_TIME = 2024-06-01T00:03:00
META_STOP

COMMENT before the maneuver
2024-06-01T00:00:00 6871.258863 0.000000 0.000000 0.000000 4.740339 5.980863
2024-06-01T00:01:00 6855.441932 284.174537 358.540891 -0.526892 4.729428 5.967097
2024-06-01T00:02:00 6808.063957 567.040862 715.431160 -1.051358 4.696745 5.925860
2024-06-01T00:03:00 6729.342943 847.296789 1069.027733 -1.570984 4.642440 5.857343

COVARIANCE_START
EPOCH = 2024-06-01T00:00:00
COV_REF_FRAME = RTN
 1.0e-06
 0.0      4.0e-06
 0.0      0.0      1.0e-06
 0.0      0.0      0.0      1.0e-12
 0.0      0.0      0.0      0.0      1.0e-12
 0.0      0.0      0.0      0.0      0.0      1.0e-12
COVARIANCE_STOP

META_START
OBJECT_NAME = DEMOSAT
OBJECT_ID = 2024-999A
CENTER_NAME = EARTH
REF_FRAME = EME2000
TIME_SYSTEM = GPS
START_TIME = 2024-06-01T00:03:00
STOP_TIME = 2024-06-01T00:05:00
META_STOP

COMMENT after the maneuver
2024-06-01T00:03:00 6729.342943 847.296789 1069.027733 -1.570984 4.650440 5.867343
2024-06-01T00:04:00 6619.641846 1124.133580 1418.312790 -2.083376 4.570932 5.767030
2024-06-01T00:05:00 6479.464939 1395.774340 1761.041780 -2.586169 4.478530 5.650446
"""

oem2 = sk.OEM.from_text(text)
for i, s in enumerate(oem2.segments):
    start, stop = s.usable_span
    print(f"segment {i}: {len(s)} states, {s.comments[0]}")
    print(f"  usable span {start} to {stop}")

a, b = oem2.segments
print(f"\ntime system {a.time_system} -> {a.time_scale}")
print(f"the label 2024-06-01T00:00:00 in GPS time is {a.time[0]} UTC")

# A time before the usable span of the first segment
try:
    oem2.interp(a.time[0] + sk.duration(seconds=30))
except ValueError as e:
    print(f"\n{e}")

# A time in the second segment is interpolated there
print(f"\nstate at {b.time[1]}: {oem2.interp(b.time[1])[:3] / 1e3} km")

cov = a.covariances[0]
print(f"\ncovariance at {cov['time']} in {cov['ref_frame']}")
print(f"position sigmas (m): {np.sqrt(np.diag(cov['cov'])[:3])}")